# Phase 4C: paired raw/SRO resume to epoch 5
Notebook: https://www.kaggle.com/code/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry
Edit and save a NEW VERSION of this existing Kaggle notebook; do not import this file as a new Kaggle notebook. Attach the exact successful prior version output, original train data, and SRO dataset. Check both progress.json files before running.


In [1]:
from pathlib import Path
import subprocess
import sys
branch = 'raw'
expected_prior_epoch = 4
source = '"""One resumable phase-4C Kaggle training segment; never reads MuSiQue dev."""\n\nimport csv\nimport hashlib\nimport inspect\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, EarlyStoppingCallback,\n                          Trainer, TrainerCallback, TrainingArguments)\n\n\nBRANCH = "raw"  # Generate a separate notebook with "sro" for the paired branch.\nMODE = "train"  # "profile" or "train"; profile does not save a checkpoint.\nTARGET_EPOCH = 5  # Train through this epoch, then stop and save for resume.\nPRIOR_ROOT = "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw"  # Mounted previous run output directory, or None for epoch 1.\nRUN_ID = "phase4c_raw_epoch5_20261124_01"\n\nBASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")\nSRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")\nOUT = Path("/kaggle/working") / f"phase4c_{BRANCH}"\nSCRATCH = Path("/kaggle/tmp") / RUN_ID\nMODEL = "microsoft/deberta-v3-large"\nREVISION = "64a8c8eab3e352a784c658aef62be1662607476f"\nSEED = 42\nSPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"\nRAW_HASH = "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"\nSRO_HASH = "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8"\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest().upper()\n\n\ndef rows(path):\n    with path.open(encoding="utf-8-sig", newline="") as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != ["text", "label"]:\n            raise ValueError(f"invalid CSV header: {path}")\n        data = list(reader)\n    if len(data) != 46610:\n        raise ValueError(f"wrong train rows: {path}")\n    return data\n\n\ndef split_key(text):\n    prefix, separator = "Sub-question: ", "\\n\\nPassages:\\n"\n    if not text.startswith(prefix) or separator not in text:\n        raise ValueError("invalid text contract")\n    return re.sub(r"\\s+", " ", text[len(prefix):].split(separator, 1)[0]).strip().casefold()\n\n\ndef split_indices(raw, sro):\n    indices = {"train": [], "validation": []}\n    groups = {}\n    for i, (a, b) in enumerate(zip(raw, sro)):\n        if a["label"] not in {"0", "1"} or a["label"] != b["label"]:\n            raise ValueError(f"label mismatch at {i}")\n        sep = "\\n\\nPassages:\\n"\n        if a["text"].split(sep, 1)[1] != b["text"].split(sep, 1)[1]:\n            raise ValueError(f"passage mismatch at {i}")\n        key = split_key(a["text"])\n        digest = hashlib.sha256(f"{SEED}\\n{key}".encode()).digest()\n        fold = "validation" if int.from_bytes(digest[:8], "big") < int(0.1 * 2**64) else "train"\n        if key in groups and groups[key] != fold:\n            raise ValueError("group leakage")\n        groups[key] = fold\n        indices[fold].append(i)\n    # The frozen indices.json was written on Windows with a CRLF final newline.\n    payload = (json.dumps(indices, separators=(",", ":")) + "\\r\\n").encode()\n    if hashlib.sha256(payload).hexdigest().upper() != SPLIT_HASH:\n        raise ValueError("frozen validation split hash mismatch")\n    if len(indices["train"]) != 42211 or len(indices["validation"]) != 4399:\n        raise ValueError("frozen validation counts mismatch")\n    return indices\n\n\nclass Encoded(torch.utils.data.Dataset):\n    def __init__(self, records, tokenizer):\n        texts = [r["text"] for r in records]\n        self.labels = [int(r["label"]) for r in records]\n        self.encodings = tokenizer(texts, max_length=512, truncation=True, padding=False)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        return {**{k: v[i] for k, v in self.encodings.items()}, "labels": self.labels[i]}\n\n\ndef metrics(prediction):\n    probs = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()\n    y = np.asarray(prediction.label_ids)\n    p = (probs >= 0.5).astype(int)\n    tp = int(((y == 1) & (p == 1)).sum())\n    fp = int(((y == 0) & (p == 1)).sum())\n    fn = int(((y == 1) & (p == 0)).sum())\n    tn = int(((y == 0) & (p == 0)).sum())\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    return {"precision": precision, "recall": recall,\n            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n            "accuracy": (tp + tn) / len(y), "flag_rate": (tn + fn) / len(y)}\n\n\nclass SegmentStop(TrainerCallback):\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch >= TARGET_EPOCH:\n            control.should_training_stop = True\n        return control\n\n\ndef checkpoint_step(path):\n    if not path.is_dir() or not re.fullmatch(r"checkpoint-\\d+", path.name):\n        raise ValueError(f"invalid checkpoint: {path}")\n    return int(path.name.split("-")[-1])\n\n\ndef model_files(checkpoint):\n    files = [p for p in checkpoint.iterdir() if p.is_file() and\n             (p.name == "config.json" or p.name == "model.safetensors" or\n              p.name == "model.safetensors.index.json" or\n              re.fullmatch(r"model-\\d+-of-\\d+\\.safetensors", p.name))]\n    if not any(p.name == "model.safetensors" or p.name.startswith("model-") for p in files):\n        raise FileNotFoundError(f"model weights missing: {checkpoint}")\n    if not any(p.name == "config.json" for p in files):\n        raise FileNotFoundError(f"model config missing: {checkpoint}")\n    return files\n\n\ndef copy_best_model_only(source, destination):\n    destination.mkdir(parents=True)\n    for path in model_files(source):\n        shutil.copy2(path, destination / path.name)\n\n\ndef patch_best_path(state_file, best_root):\n    state = json.loads(state_file.read_text())\n    name = Path(state["best_model_checkpoint"]).name\n    if not re.fullmatch(r"checkpoint-\\d+", name):\n        raise ValueError(f"invalid best checkpoint in {state_file}: {name}")\n    best = best_root / name\n    if not best.is_dir():\n        raise FileNotFoundError(best)\n    state["best_model_checkpoint"] = str(best)\n    state_file.write_text(json.dumps(state, indent=2) + "\\n")\n    return best\n\n\ndef stage_prior(prior, scratch):\n    source_root = prior / "checkpoints"\n    checkpoints = list(source_root.glob("checkpoint-*"))\n    if not checkpoints:\n        raise FileNotFoundError(f"no prior checkpoints: {source_root}")\n    latest = max(checkpoints, key=checkpoint_step)\n    target_root = scratch / "checkpoints"\n    target_root.mkdir(parents=True)\n    shutil.copytree(latest, target_root / latest.name)\n    state_file = target_root / latest.name / "trainer_state.json"\n    state = json.loads(state_file.read_text())\n    best_name = Path(state["best_model_checkpoint"]).name\n    if best_name != latest.name:\n        copy_best_model_only(source_root / best_name, target_root / best_name)\n    patch_best_path(state_file, target_root)\n    return target_root / latest.name\n\n\ndef publish_checkpoints(scratch, output, best_checkpoint):\n    source_root = scratch / "checkpoints"\n    checkpoints = list(source_root.glob("checkpoint-*"))\n    latest = max(checkpoints, key=checkpoint_step)\n    best = Path(best_checkpoint)\n    if best.parent != source_root or not best.is_dir():\n        raise ValueError(f"best checkpoint outside scratch: {best}")\n    target_root = output / "checkpoints"\n    target_root.mkdir(parents=True)\n    shutil.copytree(latest, target_root / latest.name)\n    if best != latest:\n        copy_best_model_only(best, target_root / best.name)\n    patch_best_path(target_root / latest.name / "trainer_state.json", target_root)\n    return target_root / best.name\n\n\ndef main():\n    if BRANCH not in {"raw", "sro"} or MODE not in {"profile", "train"}:\n        raise ValueError("invalid branch or mode")\n    if MODE == "train" and not 1 <= TARGET_EPOCH <= 10:\n        raise ValueError("target epoch out of range")\n    raw_path, sro_path = BASE / "deberta_fg_train.csv", SRO / "train.csv"\n    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:\n        raise ValueError("source train CSV hash mismatch")\n    raw, sro = rows(raw_path), rows(sro_path)\n    indices = split_indices(raw, sro)\n    branch_rows = raw if BRANCH == "raw" else sro\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("expected Kaggle T4 x2")\n    prior_progress = None\n    prior = None\n    if PRIOR_ROOT is None:\n        model_source = MODEL\n        source_kwargs = {"revision": REVISION}\n    else:\n        prior = Path(PRIOR_ROOT)\n        prior_progress = json.loads((prior / "progress.json").read_text())\n        if (prior_progress["branch"] != BRANCH or\n                prior_progress["split_sha256"] != SPLIT_HASH or\n                prior_progress["model_revision"] != REVISION):\n            raise ValueError("prior run branch, split or model revision mismatch")\n        model_source = prior / "checkpoints" / f"checkpoint-{prior_progress[\'global_step\']}"\n        model_files(model_source)\n        for name in ("tokenizer.json", "tokenizer_config.json", "trainer_state.json"):\n            if not (model_source / name).is_file():\n                raise FileNotFoundError(model_source / name)\n        # New Kaggle containers may have no Hugging Face cache or internet.\n        source_kwargs = {"local_files_only": True}\n    tokenizer = AutoTokenizer.from_pretrained(model_source, **source_kwargs)\n    train_records = [branch_rows[i] for i in indices["train"]]\n    validation_records = [branch_rows[i] for i in indices["validation"]]\n    train_ds = Encoded(train_records[:800] if MODE == "profile" else train_records, tokenizer)\n    val_ds = Encoded(validation_records, tokenizer) if MODE == "train" else None\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_source, **source_kwargs, num_labels=2,\n        dtype=torch.float32,\n        id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},\n        label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1})\n    # FP16 AMP keeps trainable weights/grads in FP32; otherwise GradScaler\n    # rejects FP16 gradients when unscaling on Kaggle\'s Transformers 5 image.\n    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}\n    if trainable_dtypes != {torch.float32}:\n        raise RuntimeError(f"expected FP32 trainable weights for FP16 AMP: {trainable_dtypes}")\n    if OUT.exists() or SCRATCH.exists():\n        raise FileExistsError(OUT)\n    SCRATCH.mkdir(parents=True)\n    if PRIOR_ROOT is None:\n        resume = None\n    else:\n        resume = stage_prior(prior, SCRATCH)\n    kwargs = dict(output_dir=str(SCRATCH / "checkpoints"), num_train_epochs=10,\n                  per_device_train_batch_size=4, per_device_eval_batch_size=4,\n                  gradient_accumulation_steps=2, learning_rate=2e-5, weight_decay=0.01,\n                  lr_scheduler_type="linear", fp16=True,\n                  gradient_checkpointing=True,\n                  gradient_checkpointing_kwargs={"use_reentrant": False},\n                  optim="adamw_torch", logging_steps=10,\n                  disable_tqdm=True, report_to="none", seed=SEED, data_seed=SEED,\n                  dataloader_num_workers=2, save_strategy="epoch" if MODE == "train" else "no",\n                  load_best_model_at_end=MODE == "train", metric_for_best_model="f1" if MODE == "train" else None,\n                  greater_is_better=True, save_total_limit=2,\n                  max_steps=50 if MODE == "profile" else -1)\n    supported = inspect.signature(TrainingArguments).parameters\n    # Transformers 5.20 moved ratio warmup into warmup_steps and renamed\n    # group_by_length to train_sampling_strategy. Both express the same setup.\n    if "warmup_ratio" in supported:\n        kwargs["warmup_ratio"] = 0.1\n    else:\n        kwargs["warmup_steps"] = 0.1\n    if "group_by_length" in supported:\n        kwargs["group_by_length"] = True\n    else:\n        kwargs["train_sampling_strategy"] = "group_by_length"\n    strategy_name = "eval_strategy" if "eval_strategy" in supported else "evaluation_strategy"\n    kwargs[strategy_name] = "epoch" if MODE == "train" else "no"\n    unsupported = set(kwargs) - set(supported)\n    if unsupported:\n        raise TypeError(f"unsupported TrainingArguments fields: {sorted(unsupported)}")\n    args = TrainingArguments(**kwargs)\n    callbacks = [SegmentStop()]\n    early_stopping = None\n    if MODE == "train":\n        early_stopping = EarlyStoppingCallback(early_stopping_patience=3)\n        if prior_progress:\n            early_stopping.early_stopping_patience_counter = prior_progress["early_stopping_patience_counter"]\n        callbacks.append(early_stopping)\n    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\n                      compute_metrics=metrics if MODE == "train" else None,\n                      data_collator=DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8),\n                      callbacks=callbacks)\n    started = time.monotonic()\n    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    elapsed = time.monotonic() - started\n    if MODE == "profile":\n        projected = elapsed / trainer.state.global_step * math.ceil(42211 / 16)\n        report = {"status": "profile_only", "branch": BRANCH, "steps": trainer.state.global_step,\n                  "elapsed_seconds": elapsed, "projected_one_epoch_training_seconds": projected,\n                  "under_9_hour_guard": projected < 9 * 3600,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        OUT.mkdir()\n        (OUT / "profile.json").write_text(json.dumps(report, indent=2) + "\\n")\n    else:\n        state = trainer.state\n        OUT.mkdir()\n        published_best = publish_checkpoints(SCRATCH, OUT, state.best_model_checkpoint)\n        report = {"status": "checkpoint_saved", "run_id": RUN_ID, "branch": BRANCH,\n                  "target_epoch": TARGET_EPOCH, "completed_epoch": state.epoch,\n                  "global_step": state.global_step, "best_validation_f1": state.best_metric,\n                  "best_checkpoint": str(published_best),\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "raw_sha256": RAW_HASH, "sro_sha256": SRO_HASH,\n                  "train_rows": 42211, "validation_rows": 4399,\n                  "early_stopping_patience_counter": early_stopping.early_stopping_patience_counter,\n                  "training_seconds": elapsed, "train_metrics": result.metrics,\n                  "history": state.log_history,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "progress.json").write_text(json.dumps(report, indent=2, default=str) + "\\n")\n    # SCRATCH is created by this run and is never part of the saved Kaggle output.\n    shutil.rmtree(SCRATCH)\n    print(json.dumps(report, indent=2, default=str), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
prior = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw")
if not (prior / "progress.json").is_file():
    raise FileNotFoundError(f"Attach the exact prior notebook output: {prior}")
import json
progress = json.loads((prior / "progress.json").read_text())
if progress["status"] != "checkpoint_saved" or progress["branch"] != branch:
    raise ValueError(f"Wrong prior run for {branch}: {progress}")
if progress["split_sha256"] != "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0":
    raise ValueError("Frozen split mismatch")
done = progress["completed_epoch"] >= 10 or progress["early_stopping_patience_counter"] >= 3
if done:
    if progress["completed_epoch"] > expected_prior_epoch:
        raise ValueError("Finished branch comes from a future epoch")
    import shutil
    best_name = Path(progress["best_checkpoint"]).name
    source_best = prior / "checkpoints" / best_name
    if not (source_best / "model.safetensors").is_file():
        raise FileNotFoundError(source_best / "model.safetensors")
    output = Path(f"/kaggle/working/phase4c_{branch}")
    if output.exists():
        raise FileExistsError(output)
    target_best = output / "checkpoints" / best_name
    target_best.mkdir(parents=True)
    for name in ("config.json", "model.safetensors"):
        shutil.copy2(source_best / name, target_best / name)
    progress["carried_to_epoch"] = expected_prior_epoch + 1
    progress["carried_from_run"] = str(prior)
    (output / "progress.json").write_text(json.dumps(progress, indent=2) + "\n")
else:
    if progress["completed_epoch"] != expected_prior_epoch or progress["target_epoch"] != expected_prior_epoch:
        raise ValueError("Prior run epoch mismatch")
    if progress["global_step"] != 2639 * expected_prior_epoch:
        raise ValueError("Prior run did not complete the expected steps")
    path = Path("/kaggle/working/_phase4c_resume_raw.py")
    path.write_text(source, encoding="utf-8")
    subprocess.run([sys.executable, str(path)], check=True)


In [2]:
from pathlib import Path
import subprocess
import sys
branch = 'sro'
expected_prior_epoch = 4
source = '"""One resumable phase-4C Kaggle training segment; never reads MuSiQue dev."""\n\nimport csv\nimport hashlib\nimport inspect\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, EarlyStoppingCallback,\n                          Trainer, TrainerCallback, TrainingArguments)\n\n\nBRANCH = "sro"  # Generate a separate notebook with "sro" for the paired branch.\nMODE = "train"  # "profile" or "train"; profile does not save a checkpoint.\nTARGET_EPOCH = 5  # Train through this epoch, then stop and save for resume.\nPRIOR_ROOT = "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro"  # Mounted previous run output directory, or None for epoch 1.\nRUN_ID = "phase4c_sro_epoch5_20261124_01"\n\nBASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")\nSRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")\nOUT = Path("/kaggle/working") / f"phase4c_{BRANCH}"\nSCRATCH = Path("/kaggle/tmp") / RUN_ID\nMODEL = "microsoft/deberta-v3-large"\nREVISION = "64a8c8eab3e352a784c658aef62be1662607476f"\nSEED = 42\nSPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"\nRAW_HASH = "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"\nSRO_HASH = "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8"\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest().upper()\n\n\ndef rows(path):\n    with path.open(encoding="utf-8-sig", newline="") as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != ["text", "label"]:\n            raise ValueError(f"invalid CSV header: {path}")\n        data = list(reader)\n    if len(data) != 46610:\n        raise ValueError(f"wrong train rows: {path}")\n    return data\n\n\ndef split_key(text):\n    prefix, separator = "Sub-question: ", "\\n\\nPassages:\\n"\n    if not text.startswith(prefix) or separator not in text:\n        raise ValueError("invalid text contract")\n    return re.sub(r"\\s+", " ", text[len(prefix):].split(separator, 1)[0]).strip().casefold()\n\n\ndef split_indices(raw, sro):\n    indices = {"train": [], "validation": []}\n    groups = {}\n    for i, (a, b) in enumerate(zip(raw, sro)):\n        if a["label"] not in {"0", "1"} or a["label"] != b["label"]:\n            raise ValueError(f"label mismatch at {i}")\n        sep = "\\n\\nPassages:\\n"\n        if a["text"].split(sep, 1)[1] != b["text"].split(sep, 1)[1]:\n            raise ValueError(f"passage mismatch at {i}")\n        key = split_key(a["text"])\n        digest = hashlib.sha256(f"{SEED}\\n{key}".encode()).digest()\n        fold = "validation" if int.from_bytes(digest[:8], "big") < int(0.1 * 2**64) else "train"\n        if key in groups and groups[key] != fold:\n            raise ValueError("group leakage")\n        groups[key] = fold\n        indices[fold].append(i)\n    # The frozen indices.json was written on Windows with a CRLF final newline.\n    payload = (json.dumps(indices, separators=(",", ":")) + "\\r\\n").encode()\n    if hashlib.sha256(payload).hexdigest().upper() != SPLIT_HASH:\n        raise ValueError("frozen validation split hash mismatch")\n    if len(indices["train"]) != 42211 or len(indices["validation"]) != 4399:\n        raise ValueError("frozen validation counts mismatch")\n    return indices\n\n\nclass Encoded(torch.utils.data.Dataset):\n    def __init__(self, records, tokenizer):\n        texts = [r["text"] for r in records]\n        self.labels = [int(r["label"]) for r in records]\n        self.encodings = tokenizer(texts, max_length=512, truncation=True, padding=False)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        return {**{k: v[i] for k, v in self.encodings.items()}, "labels": self.labels[i]}\n\n\ndef metrics(prediction):\n    probs = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()\n    y = np.asarray(prediction.label_ids)\n    p = (probs >= 0.5).astype(int)\n    tp = int(((y == 1) & (p == 1)).sum())\n    fp = int(((y == 0) & (p == 1)).sum())\n    fn = int(((y == 1) & (p == 0)).sum())\n    tn = int(((y == 0) & (p == 0)).sum())\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    return {"precision": precision, "recall": recall,\n            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n            "accuracy": (tp + tn) / len(y), "flag_rate": (tn + fn) / len(y)}\n\n\nclass SegmentStop(TrainerCallback):\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch >= TARGET_EPOCH:\n            control.should_training_stop = True\n        return control\n\n\ndef checkpoint_step(path):\n    if not path.is_dir() or not re.fullmatch(r"checkpoint-\\d+", path.name):\n        raise ValueError(f"invalid checkpoint: {path}")\n    return int(path.name.split("-")[-1])\n\n\ndef model_files(checkpoint):\n    files = [p for p in checkpoint.iterdir() if p.is_file() and\n             (p.name == "config.json" or p.name == "model.safetensors" or\n              p.name == "model.safetensors.index.json" or\n              re.fullmatch(r"model-\\d+-of-\\d+\\.safetensors", p.name))]\n    if not any(p.name == "model.safetensors" or p.name.startswith("model-") for p in files):\n        raise FileNotFoundError(f"model weights missing: {checkpoint}")\n    if not any(p.name == "config.json" for p in files):\n        raise FileNotFoundError(f"model config missing: {checkpoint}")\n    return files\n\n\ndef copy_best_model_only(source, destination):\n    destination.mkdir(parents=True)\n    for path in model_files(source):\n        shutil.copy2(path, destination / path.name)\n\n\ndef patch_best_path(state_file, best_root):\n    state = json.loads(state_file.read_text())\n    name = Path(state["best_model_checkpoint"]).name\n    if not re.fullmatch(r"checkpoint-\\d+", name):\n        raise ValueError(f"invalid best checkpoint in {state_file}: {name}")\n    best = best_root / name\n    if not best.is_dir():\n        raise FileNotFoundError(best)\n    state["best_model_checkpoint"] = str(best)\n    state_file.write_text(json.dumps(state, indent=2) + "\\n")\n    return best\n\n\ndef stage_prior(prior, scratch):\n    source_root = prior / "checkpoints"\n    checkpoints = list(source_root.glob("checkpoint-*"))\n    if not checkpoints:\n        raise FileNotFoundError(f"no prior checkpoints: {source_root}")\n    latest = max(checkpoints, key=checkpoint_step)\n    target_root = scratch / "checkpoints"\n    target_root.mkdir(parents=True)\n    shutil.copytree(latest, target_root / latest.name)\n    state_file = target_root / latest.name / "trainer_state.json"\n    state = json.loads(state_file.read_text())\n    best_name = Path(state["best_model_checkpoint"]).name\n    if best_name != latest.name:\n        copy_best_model_only(source_root / best_name, target_root / best_name)\n    patch_best_path(state_file, target_root)\n    return target_root / latest.name\n\n\ndef publish_checkpoints(scratch, output, best_checkpoint):\n    source_root = scratch / "checkpoints"\n    checkpoints = list(source_root.glob("checkpoint-*"))\n    latest = max(checkpoints, key=checkpoint_step)\n    best = Path(best_checkpoint)\n    if best.parent != source_root or not best.is_dir():\n        raise ValueError(f"best checkpoint outside scratch: {best}")\n    target_root = output / "checkpoints"\n    target_root.mkdir(parents=True)\n    shutil.copytree(latest, target_root / latest.name)\n    if best != latest:\n        copy_best_model_only(best, target_root / best.name)\n    patch_best_path(target_root / latest.name / "trainer_state.json", target_root)\n    return target_root / best.name\n\n\ndef main():\n    if BRANCH not in {"raw", "sro"} or MODE not in {"profile", "train"}:\n        raise ValueError("invalid branch or mode")\n    if MODE == "train" and not 1 <= TARGET_EPOCH <= 10:\n        raise ValueError("target epoch out of range")\n    raw_path, sro_path = BASE / "deberta_fg_train.csv", SRO / "train.csv"\n    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:\n        raise ValueError("source train CSV hash mismatch")\n    raw, sro = rows(raw_path), rows(sro_path)\n    indices = split_indices(raw, sro)\n    branch_rows = raw if BRANCH == "raw" else sro\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("expected Kaggle T4 x2")\n    prior_progress = None\n    prior = None\n    if PRIOR_ROOT is None:\n        model_source = MODEL\n        source_kwargs = {"revision": REVISION}\n    else:\n        prior = Path(PRIOR_ROOT)\n        prior_progress = json.loads((prior / "progress.json").read_text())\n        if (prior_progress["branch"] != BRANCH or\n                prior_progress["split_sha256"] != SPLIT_HASH or\n                prior_progress["model_revision"] != REVISION):\n            raise ValueError("prior run branch, split or model revision mismatch")\n        model_source = prior / "checkpoints" / f"checkpoint-{prior_progress[\'global_step\']}"\n        model_files(model_source)\n        for name in ("tokenizer.json", "tokenizer_config.json", "trainer_state.json"):\n            if not (model_source / name).is_file():\n                raise FileNotFoundError(model_source / name)\n        # New Kaggle containers may have no Hugging Face cache or internet.\n        source_kwargs = {"local_files_only": True}\n    tokenizer = AutoTokenizer.from_pretrained(model_source, **source_kwargs)\n    train_records = [branch_rows[i] for i in indices["train"]]\n    validation_records = [branch_rows[i] for i in indices["validation"]]\n    train_ds = Encoded(train_records[:800] if MODE == "profile" else train_records, tokenizer)\n    val_ds = Encoded(validation_records, tokenizer) if MODE == "train" else None\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_source, **source_kwargs, num_labels=2,\n        dtype=torch.float32,\n        id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},\n        label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1})\n    # FP16 AMP keeps trainable weights/grads in FP32; otherwise GradScaler\n    # rejects FP16 gradients when unscaling on Kaggle\'s Transformers 5 image.\n    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}\n    if trainable_dtypes != {torch.float32}:\n        raise RuntimeError(f"expected FP32 trainable weights for FP16 AMP: {trainable_dtypes}")\n    if OUT.exists() or SCRATCH.exists():\n        raise FileExistsError(OUT)\n    SCRATCH.mkdir(parents=True)\n    if PRIOR_ROOT is None:\n        resume = None\n    else:\n        resume = stage_prior(prior, SCRATCH)\n    kwargs = dict(output_dir=str(SCRATCH / "checkpoints"), num_train_epochs=10,\n                  per_device_train_batch_size=4, per_device_eval_batch_size=4,\n                  gradient_accumulation_steps=2, learning_rate=2e-5, weight_decay=0.01,\n                  lr_scheduler_type="linear", fp16=True,\n                  gradient_checkpointing=True,\n                  gradient_checkpointing_kwargs={"use_reentrant": False},\n                  optim="adamw_torch", logging_steps=10,\n                  disable_tqdm=True, report_to="none", seed=SEED, data_seed=SEED,\n                  dataloader_num_workers=2, save_strategy="epoch" if MODE == "train" else "no",\n                  load_best_model_at_end=MODE == "train", metric_for_best_model="f1" if MODE == "train" else None,\n                  greater_is_better=True, save_total_limit=2,\n                  max_steps=50 if MODE == "profile" else -1)\n    supported = inspect.signature(TrainingArguments).parameters\n    # Transformers 5.20 moved ratio warmup into warmup_steps and renamed\n    # group_by_length to train_sampling_strategy. Both express the same setup.\n    if "warmup_ratio" in supported:\n        kwargs["warmup_ratio"] = 0.1\n    else:\n        kwargs["warmup_steps"] = 0.1\n    if "group_by_length" in supported:\n        kwargs["group_by_length"] = True\n    else:\n        kwargs["train_sampling_strategy"] = "group_by_length"\n    strategy_name = "eval_strategy" if "eval_strategy" in supported else "evaluation_strategy"\n    kwargs[strategy_name] = "epoch" if MODE == "train" else "no"\n    unsupported = set(kwargs) - set(supported)\n    if unsupported:\n        raise TypeError(f"unsupported TrainingArguments fields: {sorted(unsupported)}")\n    args = TrainingArguments(**kwargs)\n    callbacks = [SegmentStop()]\n    early_stopping = None\n    if MODE == "train":\n        early_stopping = EarlyStoppingCallback(early_stopping_patience=3)\n        if prior_progress:\n            early_stopping.early_stopping_patience_counter = prior_progress["early_stopping_patience_counter"]\n        callbacks.append(early_stopping)\n    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\n                      compute_metrics=metrics if MODE == "train" else None,\n                      data_collator=DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8),\n                      callbacks=callbacks)\n    started = time.monotonic()\n    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    elapsed = time.monotonic() - started\n    if MODE == "profile":\n        projected = elapsed / trainer.state.global_step * math.ceil(42211 / 16)\n        report = {"status": "profile_only", "branch": BRANCH, "steps": trainer.state.global_step,\n                  "elapsed_seconds": elapsed, "projected_one_epoch_training_seconds": projected,\n                  "under_9_hour_guard": projected < 9 * 3600,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        OUT.mkdir()\n        (OUT / "profile.json").write_text(json.dumps(report, indent=2) + "\\n")\n    else:\n        state = trainer.state\n        OUT.mkdir()\n        published_best = publish_checkpoints(SCRATCH, OUT, state.best_model_checkpoint)\n        report = {"status": "checkpoint_saved", "run_id": RUN_ID, "branch": BRANCH,\n                  "target_epoch": TARGET_EPOCH, "completed_epoch": state.epoch,\n                  "global_step": state.global_step, "best_validation_f1": state.best_metric,\n                  "best_checkpoint": str(published_best),\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "raw_sha256": RAW_HASH, "sro_sha256": SRO_HASH,\n                  "train_rows": 42211, "validation_rows": 4399,\n                  "early_stopping_patience_counter": early_stopping.early_stopping_patience_counter,\n                  "training_seconds": elapsed, "train_metrics": result.metrics,\n                  "history": state.log_history,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "progress.json").write_text(json.dumps(report, indent=2, default=str) + "\\n")\n    # SCRATCH is created by this run and is never part of the saved Kaggle output.\n    shutil.rmtree(SCRATCH)\n    print(json.dumps(report, indent=2, default=str), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
prior = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro")
if not (prior / "progress.json").is_file():
    raise FileNotFoundError(f"Attach the exact prior notebook output: {prior}")
import json
progress = json.loads((prior / "progress.json").read_text())
if progress["status"] != "checkpoint_saved" or progress["branch"] != branch:
    raise ValueError(f"Wrong prior run for {branch}: {progress}")
if progress["split_sha256"] != "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0":
    raise ValueError("Frozen split mismatch")
done = progress["completed_epoch"] >= 10 or progress["early_stopping_patience_counter"] >= 3
if done:
    if progress["completed_epoch"] > expected_prior_epoch:
        raise ValueError("Finished branch comes from a future epoch")
    import shutil
    best_name = Path(progress["best_checkpoint"]).name
    source_best = prior / "checkpoints" / best_name
    if not (source_best / "model.safetensors").is_file():
        raise FileNotFoundError(source_best / "model.safetensors")
    output = Path(f"/kaggle/working/phase4c_{branch}")
    if output.exists():
        raise FileExistsError(output)
    target_best = output / "checkpoints" / best_name
    target_best.mkdir(parents=True)
    for name in ("config.json", "model.safetensors"):
        shutil.copy2(source_best / name, target_best / name)
    progress["carried_to_epoch"] = expected_prior_epoch + 1
    progress["carried_from_run"] = str(prior)
    (output / "progress.json").write_text(json.dumps(progress, indent=2) + "\n")
else:
    if progress["completed_epoch"] != expected_prior_epoch or progress["target_epoch"] != expected_prior_epoch:
        raise ValueError("Prior run epoch mismatch")
    if progress["global_step"] != 2639 * expected_prior_epoch:
        raise ValueError("Prior run did not complete the expected steps")
    path = Path("/kaggle/working/_phase4c_resume_sro.py")
    path.write_text(source, encoding="utf-8")
    subprocess.run([sys.executable, str(path)], check=True)


Loading weights: 100%|██████████| 394/394 [00:24<00:00, 16.10it/s]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '1.358', 'grad_norm': '6.465', 'learning_rate': '1.333e-05', 'epoch': '4.002'}
{'loss': '1.319', 'grad_norm': '4.081', 'learning_rate': '1.332e-05', 'epoch': '4.005'}
{'loss': '1.1', 'grad_norm': '4.466', 'learning_rate': '1.331e-05', 'epoch': '4.009'}
{'loss': '0.9647', 'grad_norm': '16.96', 'learning_rate': '1.331e-05', 'epoch': '4.013'}
{'loss': '0.9237', 'grad_norm': '11.42', 'learning_rate': '1.33e-05', 'epoch': '4.017'}
{'loss': '0.7994', 'grad_norm': '7.048', 'learning_rate': '1.329e-05', 'epoch': '4.02'}
{'loss': '1.06', 'grad_norm': '5.14', 'learning_rate': '1.328e-05', 'epoch': '4.024'}
{'loss': '1.163', 'grad_norm': '3.864', 'learning_rate': '1.327e-05', 'epoch': '4.028'}
{'loss': '1.193', 'grad_norm': '10.49', 'learning_rate': '1.326e-05', 'epoch': '4.032'}
{'loss': '1.276', 'grad_norm': '6.317', 'learning_rate': '1.326e-05', 'epoch': '4.036'}
{'loss': '0.9717', 'grad_norm': '5.696', 'learning_rate': '1.325e-05', 'epoch': '4.039'}
{'loss': '0.8348', 'grad_norm': '8

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.81s/it]


{'train_runtime': '1.395e+04', 'train_samples_per_second': '3.025', 'train_steps_per_second': '0.189', 'train_loss': '1.175', 'epoch': '5'}
{
  "status": "checkpoint_saved",
  "run_id": "phase4c_sro_epoch5_20261124_01",
  "branch": "sro",
  "target_epoch": 5,
  "completed_epoch": 5.0,
  "global_step": 13195,
  "best_validation_f1": 0.7822410147991543,
  "best_checkpoint": "/kaggle/working/phase4c_sro/checkpoints/checkpoint-5278",
  "split_sha256": "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0",
  "model_revision": "64a8c8eab3e352a784c658aef62be1662607476f",
  "raw_sha256": "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D",
  "sro_sha256": "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8",
  "train_rows": 42211,
  "validation_rows": 4399,
  "early_stopping_patience_counter": 3,
  "training_seconds": 13959.155019393,
  "train_metrics": {
    "train_runtime": 13954.242,
    "train_samples_per_second": 3.025,
    "train_steps_per_second